In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
BASE = "/content/drive/MyDrive/Just_Advisor_Ai"

MODEL_PATH = f"{BASE}/models/legalbert/final"
TRANSCRIPT_PATH = f"{BASE}/transcripts/case1.json"
OUTPUT_PATH = f"{BASE}/structured_arguments/case1_structured.json"


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

labels = ["Claim", "Evidence", "LegalRule", "Rebuttal", "Conclusion"]

print("Model loaded.")


Model loaded.


In [4]:
import json

with open(TRANSCRIPT_PATH) as f:
    transcript = json.load(f)

transcript


[{'speaker': 'Lawyer_A',
  'time': '14:00:34',
  'text': 'The accused violated Section 420 IPC.'},
 {'speaker': 'Lawyer_B',
  'time': '14:00:34',
  'text': 'There is no fraudulent intention.'},
 {'speaker': 'Lawyer_A',
  'time': '14:00:34',
  'text': 'Bank records prove deception.'}]

In [9]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
from nltk.tokenize import sent_tokenize
sentences = []

for turn in transcript:
    for s in sent_tokenize(turn["text"]):
        sentences.append({
            "speaker": turn["speaker"],
            "sentence": s
        })

sentences


[{'speaker': 'Lawyer_A', 'sentence': 'The accused violated Section 420 IPC.'},
 {'speaker': 'Lawyer_B', 'sentence': 'There is no fraudulent intention.'},
 {'speaker': 'Lawyer_A', 'sentence': 'Bank records prove deception.'}]

In [12]:
def classify(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    return labels[logits.argmax().item()]

classified = []

for s in sentences:
    role = classify(s["sentence"])
    classified.append({
        "speaker": s["speaker"],
        "sentence": s["sentence"],
        "role": role
    })

classified


[{'speaker': 'Lawyer_A',
  'sentence': 'The accused violated Section 420 IPC.',
  'role': 'Claim'},
 {'speaker': 'Lawyer_B',
  'sentence': 'There is no fraudulent intention.',
  'role': 'Rebuttal'},
 {'speaker': 'Lawyer_A',
  'sentence': 'Bank records prove deception.',
  'role': 'Evidence'}]

In [13]:
structured = {}

for item in classified:
    speaker = item["speaker"]
    role = item["role"]
    sentence = item["sentence"]

    if speaker not in structured:
        structured[speaker] = {
            "claims": [],
            "evidence": [],
            "laws": [],
            "rebuttals": [],
            "conclusions": []
        }

    if role == "Claim":
        structured[speaker]["claims"].append(sentence)
    elif role == "Evidence":
        structured[speaker]["evidence"].append(sentence)
    elif role == "LegalRule":
        structured[speaker]["laws"].append(sentence)
    elif role == "Rebuttal":
        structured[speaker]["rebuttals"].append(sentence)
    elif role == "Conclusion":
        structured[speaker]["conclusions"].append(sentence)


In [14]:
import os
os.makedirs(f"{BASE}/structured_arguments", exist_ok=True)

with open(OUTPUT_PATH, "w") as f:
    json.dump(structured, f, indent=2)

print("Saved to:", OUTPUT_PATH)


Saved to: /content/drive/MyDrive/Just_Advisor_Ai/structured_arguments/case1_structured.json
